In [ ]:
!pip install gensim
!pip install nltk
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import nltk
from nltk.corpus import wordnet, stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from collections import Counter

# Download NLTK data (run once)
nltk.download('punkt'); nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('omw-1.4'); nltk.download('punkt_tab')

# Load Data
train_df = pd.read_csv("mental_health_training.csv")
test_df = pd.read_csv("mental_health_test.csv")

# --- TASK 1: TF-IDF Vocabulary ---
print("--- Task 1: TF-IDF ---")
vectorizer = TfidfVectorizer(stop_words='english')
X_train = vectorizer.fit_transform(train_df['Text'])
X_test = vectorizer.transform(test_df['Text'])

print("Vocabulary:", vectorizer.get_feature_names_out()[:10]) # Printing first 10
print("Train Matrix Shape:", X_train.shape)
print("Test Matrix Shape:", X_test.shape)

# --- TASK 2: WordNet & Word2Vec ---
print("\n--- Task 2: WordNet & Word2Vec ---")
# Combine text to find most frequent word
full_text = pd.concat([train_df['Text'], test_df['Text']])
stop_words = set(stopwords.words("english"))
all_tokens = []
sentences = []

for text in full_text:
    tokens = word_tokenize(str(text).lower())
    clean = [w for w in tokens if w.isalnum() and w not in stop_words]
    all_tokens.extend(clean)
    sentences.append(clean)

# Find Highest Frequency Word
top_word = Counter(all_tokens).most_common(1)[0][0]
print(f"Highest Frequency Word: '{top_word}'")

# Print Synsets
synsets = wordnet.synsets(top_word)
if synsets:
    print(f"Definition: {synsets[0].definition()}")

# Train Word2Vec and Print Embedding
w2v_model = Word2Vec(sentences, vector_size=10, window=5, min_count=1, workers=1)
print("Embedding (first 10):", w2v_model.wv[top_word])

# --- TASK 3: Doc2Vec Classification ---
print("\n--- Task 3: Doc2Vec Classification ---")
# Tag documents
train_tagged = [TaggedDocument(words=word_tokenize(str(t).lower()), tags=[str(i)]) for i, t in enumerate(train_df['Text'])]
test_tokens = [word_tokenize(str(t).lower()) for t in test_df['Text']]

# Train Doc2Vec
d2v_model = Doc2Vec(vector_size=50, window=2, min_count=1, workers=1, epochs=30)
d2v_model.build_vocab(train_tagged)
d2v_model.train(train_tagged, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

# Prepare Vectors
X_train_vec = [d2v_model.infer_vector(doc.words) for doc in train_tagged]
X_test_vec = [d2v_model.infer_vector(tokens) for tokens in test_tokens]

# Train Classifier & Predict
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, train_df['Label'])
y_pred = clf.predict(X_test_vec)

# Since the 'test_df' DataFrame does not contain a 'Label' column,
# we cannot calculate metrics like F1 Score for evaluation.
# However, we have successfully generated predictions for the test set.
print("Predictions for test_df:", y_pred[:10]) # Displaying first 10 predictions

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


--- Task 1: TF-IDF ---
Vocabulary: ['00pm' '10' '100' '100lbs' '100mg' '10liters' '10mg' '10mn' '11' '110']
Train Matrix Shape: (310, 4623)
Test Matrix Shape: (82, 4623)

--- Task 2: WordNet & Word2Vec ---
Highest Frequency Word: 'like'
Definition: a similar kind; ,
Embedding (first 10): [ 0.93932635 -1.0131915   1.9633178   0.48529232  1.5212784   0.69541365
  2.9567492   1.3614092  -2.4056902  -1.5557828 ]

--- Task 3: Doc2Vec Classification ---
Predictions for test_df: ['Anxiety' 'Anxiety' 'Anxiety' 'Depression' 'Anxiety' 'Anxiety'
 'Depression' 'Depression' 'Anxiety' 'Anxiety']


In [ ]:
import pandas as pd
import nltk
from nltk.corpus import wordnet, stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from collections import Counter

# 0. Setup
nltk.download(['punkt', 'stopwords', 'wordnet', 'omw-1.4'], quiet=True)
stop_words = set(stopwords.words("english"))

train = pd.read_csv("mental_health_training.csv")
test = pd.read_csv("mental_health_test.csv")

# --- TASK 1: TF-IDF Vocabularies ---
print("\n--- Task 1 ---")
# Fit separate vectorizers to get specific vocabularies for both as requested
vec_train = TfidfVectorizer(stop_words='english').fit(train['Text'])
vec_test = TfidfVectorizer(stop_words='english').fit(test['Text'])

print(f"Train Vocab (First 5): {list(vec_train.vocabulary_.keys())[:5]}")
print(f"Test Vocab (First 5):  {list(vec_test.vocabulary_.keys())[:5]}")

# --- TASK 2: WordNet & Word2Vec (Size 10) ---
print("\n--- Task 2 ---")
# Tokenize and clean all text in one go
all_text = pd.concat([train['Text'], test['Text']])
tokens_list = [[w for w in word_tokenize(str(t).lower()) if w.isalpha() and w not in stop_words] for t in all_text]

# Find most frequent word
flat_tokens = [item for sublist in tokens_list for item in sublist]
top_word = Counter(flat_tokens).most_common(1)[0][0]
print(f"Top Word: {top_word}")

# Print Synsets
print(f"Synsets: {wordnet.synsets(top_word)}")

# Train Word2Vec (Size 10)
w2v = Word2Vec(tokens_list, vector_size=10, min_count=1)
print(f"Embedding: {w2v.wv[top_word]}")

# --- TASK 3: Doc2Vec & Classification ---
print("\n--- Task 3 ---")
# Tag Training Docs
train_tagged = [TaggedDocument(words=t, tags=[i]) for i, t in enumerate(tokens_list[:len(train)])]
test_tokens = tokens_list[len(train):]

# Train Doc2Vec
d2v = Doc2Vec(train_tagged, vector_size=50, epochs=30, min_count=1)

# Infer Vectors
X_train_d2v = [d2v.infer_vector(d.words) for d in train_tagged]
X_test_d2v  = [d2v.infer_vector(t) for t in test_tokens]

# Predict
clf = LogisticRegression(max_iter=1000).fit(X_train_d2v, train['Label'])
preds = clf.predict(X_test_d2v)

# Calculate F1 only if Test data has labels, otherwise just print predictions
if 'Label' in test.columns:
    print(f"F1 Score: {f1_score(test['Label'], preds, average='weighted')}")
else:
    print(f"Test labels missing. First 10 Predictions: {preds[:10]}")

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from collections import Counter

# 1. SETUP
nltk.download('punkt'); nltk.download('wordnet')
train = pd.read_csv("mental_health_training.csv")
test = pd.read_csv("mental_health_test.csv")

# 2. TF-IDF VOCABULARY
print("--- Task 1 ---")
# Just fit the vectorizer and print the keys
tfidf = TfidfVectorizer()
tfidf.fit(train['Text'])
print("Train Vocab:", list(tfidf.vocabulary_.keys())[:5])

tfidf.fit(test['Text'])
print("Test Vocab:", list(tfidf.vocabulary_.keys())[:5])

# 3. WORDNET & WORD2VEC
print("--- Task 2 ---")
# Create a big list of all words for counting
all_words = []
sentences = [] # Needed for Word2Vec

for text in train['Text']:
    words = word_tokenize(text.lower())
    all_words.extend(words)
    sentences.append(words)

# Find top word
top_word = Counter(all_words).most_common(1)[0][0]
print("Top word:", top_word)

# Print Synsets
from nltk.corpus import wordnet
print("Synsets:", wordnet.synsets(top_word))

# Train Word2Vec
w2v = Word2Vec(sentences, vector_size=10, min_count=1)
print("Vector:", w2v.wv[top_word])

# 4. DOC2VEC CLASSIFICATION
print("--- Task 3 ---")
# Prepare data for Doc2Vec
train_docs = [TaggedDocument(words=word_tokenize(t.lower()), tags=[i]) for i, t in enumerate(train['Text'])]

# Train Model
d2v = Doc2Vec(train_docs, vector_size=50, epochs=20, min_count=1)

# Turn text into vectors
X_train = [d2v.infer_vector(doc.words) for doc in train_docs]
X_test = [d2v.infer_vector(word_tokenize(t.lower())) for t in test['Text']]

# Classify
clf = LogisticRegression()
clf.fit(X_train, train['Label'])
y_pred = clf.predict(X_test)

print("Predictions:", y_pred[:5])
# print(f1_score(test['Label'], y_pred)) # Uncomment if test CSV has labels

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


--- Task 1 ---
Train Vocab: ['about', 'weeks', 'ago', 'told', 'my']
Test Vocab: ['don', 'understand', 'whats', 'wrong', 'with']
--- Task 2 ---
Top word: i
Synsets: [Synset('iodine.n.01'), Synset('one.n.01'), Synset('i.n.03'), Synset('one.s.01')]
Vector: [ 1.2866244  -1.2935579   2.820278    0.6640071  -0.1153494   0.33300188
  3.5163724   1.3794303  -3.2942803  -2.01341   ]
--- Task 3 ---
Predictions: ['Anxiety' 'Anxiety' 'Anxiety' 'Anxiety' 'Anxiety']


In [ ]:
import numpy as np

# Save predictions to a labels.txt file
np.savetxt('labels.txt', y_pred, fmt='%s')

In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from collections import Counter

# 1. SETUP
nltk.download('punkt'); nltk.download('wordnet')
train = pd.read_csv("mental_health_training.csv")
test = pd.read_csv("mental_health_test.csv")

# 2. TF-IDF VOCABULARY
print("--- Task 1 ---")
tfidf = TfidfVectorizer()
tfidf.fit(train['Text'])
print("Train Vocab:", list(tfidf.vocabulary_.keys())[:5])

tfidf.fit(test['Text'])
print("Test Vocab:", list(tfidf.vocabulary_.keys())[:5])

# 3. WORDNET & WORD2VEC
print("--- Task 2 ---")
all_words = []
sentences = []

for text in train['Text']:
    words = word_tokenize(text.lower())
    all_words.extend(words)
    sentences.append(words)

# Top word
top_word = Counter(all_words).most_common(1)[0][0]
print("Top word:", top_word)

# Synsets & Embedding
print("Synsets:", wordnet.synsets(top_word))
w2v = Word2Vec(sentences, vector_size=10, min_count=1)
print("Vector:", w2v.wv[top_word])

# 4. DOC2VEC & OUTPUT FILE
print("--- Task 3 ---")
# Train Doc2Vec
train_docs = [TaggedDocument(words=word_tokenize(t.lower()), tags=[i]) for i, t in enumerate(train['Text'])]
d2v = Doc2Vec(train_docs, vector_size=50, epochs=20, min_count=1)

# Get Vectors
X_train = [d2v.infer_vector(doc.words) for doc in train_docs]
X_test = [d2v.infer_vector(word_tokenize(t.lower())) for t in test['Text']]

# Predict
clf = LogisticRegression()
clf.fit(X_train, train['Label'])
predictions = clf.predict(X_test)

# SAVE TO labels.txt
pd.DataFrame(predictions).to_csv("labels.txt", index=False, header=False)
print("Saved to labels.txt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


--- Task 1 ---
Train Vocab: ['about', 'weeks', 'ago', 'told', 'my']
Test Vocab: ['don', 'understand', 'whats', 'wrong', 'with']
--- Task 2 ---
Top word: i
Synsets: [Synset('iodine.n.01'), Synset('one.n.01'), Synset('i.n.03'), Synset('one.s.01')]
Vector: [ 1.4260086  -1.2846816   2.7715776   0.6414391  -0.12737316  0.3312972
  3.4723446   1.3677675  -3.2232194  -2.0099611 ]
--- Task 3 ---
Saved to labels.txt
